## Install Libraries

Runs the **dense baseline end-to-end evaluation**: read the candidate file
written by `baseline/retrieval.ipynb`, keep the stored Top-1 as the prediction
(the baseline has no re-ranker), report R@1/R@5/accuracy and write the metrics
JSON.

This mirrors `GRF/e2e.ipynb` stage for stage, so both payloads can be compared
row by row; the only difference is that the re-ranking step is the identity.
The notebook never loads the encoder, the vector index or the KB artifact, so
`tqdm` is the only dependency.


In [ ]:
%pip install "tqdm>=4.66"


## Import Libraries

Third-party imports only. The shared `common` package lives inside the
repository clone, so project modules are imported at the end of the
repository-setup stage below.


In [ ]:
from __future__ import annotations

import gc
import json
import subprocess
import sys
import time
from contextlib import contextmanager
from dataclasses import asdict
from pathlib import Path

from tqdm.auto import tqdm

print("Python:", sys.version.split()[0])
print("tqdm:", __import__("tqdm").__version__)


@contextmanager
def progress_bar(total, desc: str, unit: str = "item"):
    """Drive a tqdm bar from a progress_callback(done, total)."""
    bar = tqdm(total=total, desc=desc, unit=unit)

    def report(done: int, total=None) -> None:
        if total is not None and total != bar.total:
            bar.total = total
        bar.update(done - bar.n)

    try:
        yield report
    finally:
        bar.close()


## Constants and Configuration

Set `DRIVE_DATA_DIR` once; the two paths below derive from it:

- `CANDIDATES_PATH`: the JSONL file written by `baseline/retrieval.ipynb`,
- `RESULTS_OUTPUT_PATH`: the end-to-end metrics written by this notebook.

No encoder, index or KB path is configured, because the baseline has nothing
to re-rank with: the Top-1 candidate stored in the file is the prediction.


In [ ]:
# ---------------------------------------------------------------------------
# Git repository (REQUIRED: replace the placeholder with your repository URL)
# ---------------------------------------------------------------------------
GIT_REPO_URL = "https://github.com/minhhhduc/bio-linking-.git"
GIT_BRANCH = "main"                                             # TODO
REPO_CLONE_DIR = Path("/content/bio_linking")

# ---------------------------------------------------------------------------
# Google Drive input and output (REQUIRED: set DRIVE_DATA_DIR below)
# ---------------------------------------------------------------------------
MOUNT_DRIVE = True
DRIVE_MOUNT_POINT = Path("/content/drive")
DRIVE_ROOT = DRIVE_MOUNT_POINT / "MyDrive"
DRIVE_DATA_DIR = DRIVE_ROOT / "REPLACE_ME"                      # TODO: set once
CANDIDATES_PATH = DRIVE_DATA_DIR / "grf_baseline_candidates.jsonl"  # from retrieval.ipynb
RESULTS_OUTPUT_PATH = DRIVE_DATA_DIR / "grf_baseline_e2e_results.json"

# ---------------------------------------------------------------------------
# Runtime
# ---------------------------------------------------------------------------
_START = time.time()


def log(message: str) -> None:
    print(f"[{time.strftime('%H:%M:%S')} +{time.time() - _START:6.1f}s] {message}", flush=True)


def require_configured(value, name: str):
    text = str(value)
    if "REPLACE_ME" in text:
        raise ValueError(f"{name} is still a placeholder: {text!r}. Set it in the constants cell.")
    return value


log("Configuration loaded.")
print(f"Repository   : {GIT_REPO_URL} ({GIT_BRANCH})")
print(f"Candidates   : {CANDIDATES_PATH}")
print(f"Results out  : {RESULTS_OUTPUT_PATH}")
print("Re-rank      : identity (baseline, no re-ranker)")


## Stage 1 — Clone or Update the Code Repository

Colab starts from an empty runtime, so the notebook clones the code repository
and imports the shared modules from `source/common/`. Re-running this cell
updates the existing clone instead of downloading it again.


In [ ]:
repo_url = str(require_configured(GIT_REPO_URL, "GIT_REPO_URL"))


def _git(*args: str) -> None:
    log("git " + " ".join(args))
    try:
        subprocess.run(["git", *args], check=True)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(f"git {args[0]} failed with exit code {exc.returncode}.") from exc


if REPO_CLONE_DIR.exists() and not (REPO_CLONE_DIR / ".git").exists():
    raise RuntimeError(
        f"{REPO_CLONE_DIR} exists but is not a git clone. "
        "Remove it or point REPO_CLONE_DIR elsewhere."
    )

if (REPO_CLONE_DIR / ".git").exists():
    log(f"Repository already present at {REPO_CLONE_DIR}; updating branch {GIT_BRANCH!r}.")
    _git("-C", str(REPO_CLONE_DIR), "fetch", "--depth", "1", "origin", GIT_BRANCH)
    _git("-C", str(REPO_CLONE_DIR), "checkout", GIT_BRANCH)
    _git("-C", str(REPO_CLONE_DIR), "pull", "--ff-only", "origin", GIT_BRANCH)
else:
    log(f"Cloning {repo_url} (branch {GIT_BRANCH!r}) into {REPO_CLONE_DIR} ...")
    REPO_CLONE_DIR.parent.mkdir(parents=True, exist_ok=True)
    _git("clone", "--depth", "1", "--branch", GIT_BRANCH, repo_url, str(REPO_CLONE_DIR))

SOURCE_DIR = REPO_CLONE_DIR / "source"
if not (SOURCE_DIR / "common" / "__init__.py").is_file():
    raise FileNotFoundError(
        f"{SOURCE_DIR / 'common' / '__init__.py'} is missing; check GIT_BRANCH and the repository layout."
    )
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

from common.candidates import load_candidates
from common.evaluator import metrics_from_responses

log(f"Repository ready at {REPO_CLONE_DIR}; common imported from {SOURCE_DIR}.")


## Stage 2 — Mount Google Drive and Locate the Candidate File

The candidate file and the results output live on Drive, not in the repository.
This stage mounts Drive and checks the input plus the output path before any
record is parsed.


In [ ]:
if MOUNT_DRIVE:
    try:
        from google.colab import drive
    except ImportError as exc:  # running outside Colab
        raise RuntimeError(
            "Google Colab is required to mount Drive. Run this notebook in Colab, "
            "or set MOUNT_DRIVE = False only if the inputs are already local to the runtime."
        ) from exc
    drive.mount(str(DRIVE_MOUNT_POINT))

candidates_path = Path(require_configured(CANDIDATES_PATH, "CANDIDATES_PATH"))
results_path = Path(require_configured(RESULTS_OUTPUT_PATH, "RESULTS_OUTPUT_PATH"))

if not candidates_path.is_file():
    raise FileNotFoundError(
        f"Candidate file not found: {candidates_path}. Run baseline/retrieval.ipynb first."
    )
if candidates_path.suffix.lower() != ".jsonl":
    raise ValueError(f"CANDIDATES_PATH must end in .jsonl, got {candidates_path.name!r}.")
if results_path.suffix.lower() != ".json":
    raise ValueError(f"RESULTS_OUTPUT_PATH must end in .json, got {results_path.name!r}.")
log(f"Candidates: {candidates_path} ({candidates_path.stat().st_size / 1024 ** 2:.2f} MB)")
log(f"Results out: {results_path}")


## Stage 3 — Load Saved Candidates

`load_candidates` restores the gold samples and the retrieval responses from the
single JSONL file. Nothing is re-encoded or re-retrieved. It reports each record
through `progress_callback`, which drives the bar.


In [ ]:
log(f"Loading candidates from {candidates_path} ...")
started = time.time()
with progress_bar(None, "Loading candidate records", "record") as report:
    samples, responses = load_candidates(candidates_path, progress_callback=report)
log(
    f"Loaded {len(samples):,} samples and {len(responses):,} responses "
    f"in {time.time() - started:.1f}s."
)
if not samples:
    raise ValueError(f"{candidates_path} contains no candidate records.")
empty = sum(1 for response in responses if not response.candidates)
log(f"Mentions with candidates: {len(responses) - empty:,}; without: {empty:,}.")


## Stage 4 — Identity Re-ranking

The baseline has no re-ranker: the prediction is the Top-1 candidate that
`baseline/retrieval.ipynb` stored, exactly what `GRF/e2e.ipynb` does with
`PickTopOne`. This stage only proves the file agrees with itself, so a corrupt or
hand-edited candidate file fails here instead of silently skewing the accuracy.


In [ ]:
without_candidates = [
    index for index, response in enumerate(responses) if not response.candidates
]
if without_candidates:
    raise ValueError(
        f"{len(without_candidates):,} record(s) have no candidates; "
        f"first at record {without_candidates[0]}."
    )

mismatched = [
    index
    for index, response in enumerate(responses)
    if response.cui != response.candidates[0].cui
]
if mismatched:
    raise ValueError(
        f"{len(mismatched):,} record(s) store a cui that is not the Top-1 candidate; "
        f"first at record {mismatched[0]}."
    )

reranked = responses
log(f"Identity re-ranking: Top-1 kept for {len(reranked):,} mentions (no model call).")


## Stage 5 — End-to-End Metrics

R@1/R@5 come from the untruncated retrieval ranking restored with the candidate
file; accuracy is the fraction of mentions whose Top-1 equals the gold CUI. With
identity re-ranking the accuracy matches the retrieval accuracy, which is the
number the GRF configurations must beat (architecture §10).


In [ ]:
overall, by_label = metrics_from_responses(samples, reranked)


def _row(label, metrics) -> str:
    return (
        f"{str(label):<16}{metrics.n:>7}"
        f"{100 * metrics.recall_at_1:>10.2f}{100 * metrics.recall_at_5:>10.2f}"
        f"{100 * metrics.accuracy:>10.2f}"
    )


print(f"{'label':<16}{'n':>7}{'R@1 %':>10}{'R@5 %':>10}{'acc %':>10}")
print(_row("ALL", overall))
for label, metrics in by_label.items():
    print(_row(label, metrics))


## Stage 6 — Save End-to-End Results

Write the candidate-file provenance, the (identity) re-ranker and the metrics to
the single configured `RESULTS_OUTPUT_PATH`, using the same payload shape as
`GRF/e2e.ipynb`, then read it back to confirm the file is usable.


In [ ]:
payload = {
    "candidates_file": str(candidates_path),
    "n_samples": len(samples),
    "rerank": "identity",
    "rerank_failures": 0,
    "overall": asdict(overall),
    "by_label": {str(label): asdict(metrics) for label, metrics in by_label.items()},
}

results_path.parent.mkdir(parents=True, exist_ok=True)
log(f"Saving results to {results_path} ...")
results_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
saved = json.loads(results_path.read_text(encoding="utf-8"))
assert saved["overall"] == payload["overall"]
log(
    f"Saved and verified {results_path} "
    f"({results_path.stat().st_size / 1024 ** 2:.2f} MB)."
)
log("Baseline end-to-end evaluation complete.")


## Stage 7 — Release Memory

Only the parsed samples and responses are held, so dropping them and collecting
garbage is enough; no model weights or index were ever loaded.


In [ ]:
for name in ("samples", "responses", "reranked", "payload", "saved"):
    globals().pop(name, None)
gc.collect()

log("Baseline E2E memory released (samples, responses).")
